<a href="https://colab.research.google.com/github/dhiaCodes/autocomplete_google_places_widget/blob/main/Are_Medical_AI_Explanations_Trustworthy_Under_Distribution_Shift%3F_A_Faithfulness_Benchmark_of_Post_hoc_XAI_Methods_on_Medical_Image_Classifiers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1 — Install all dependencies
# ============================================================
!pip install medmnist timm captum lime shap albumentations scikit-image matplotlib seaborn pandas "numpy>=2.0.0" -q

In [ ]:
# ============================================================
# CELL 2 — Imports & Global Config
# ============================================================
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models

import timm
import medmnist
from medmnist import INFO, Evaluator

import albumentations as A
from albumentations.pytorch import ToTensorV2

from captum.attr import LayerGradCam, IntegratedGradients, NoiseTunnel
import lime
from lime import lime_image
import shap

from skimage.metrics import structural_similarity as ssim
from scipy.stats import spearmanr

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# ── Global Settings ──────────────────────────────────────────
IMG_SIZE   = 64          # resize all MedMNIST to 64x64
BATCH_SIZE = 64
EPOCHS     = 10          # enough for a benchmark study
LR         = 1e-3
NUM_WORKERS = 2

# Datasets to benchmark
DATASETS = {
    "PathMNIST":  "pathmnist",
    "ChestMNIST": "chestmnist",
    "DermaMNIST": "dermamnist",
}

# XAI methods we will apply
XAI_METHODS = ["GradCAM", "IntegratedGradients", "LIME", "SHAP"]

# Corruption types
CORRUPTION_TYPES = ["clean", "gaussian_noise", "motion_blur",
                    "contrast_shift", "brightness_shift", "salt_pepper"]

print("✅ Config ready.")

In [ ]:
# ============================================================
# CELL 3 — Load MedMNIST datasets & train 3 classifiers
#           per dataset. Models saved to ./checkpoints/
# Runtime: ~25–40 min on GPU, ~90 min on CPU
# ============================================================

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results",     exist_ok=True)

# ── Transform ────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# ── Model factory ────────────────────────────────────────────
def build_model(model_name: str, num_classes: int) -> nn.Module:
    if model_name == "ResNet18":
        m = models.resnet18(weights="IMAGENET1K_V1")
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == "EfficientNetB0":
        m = timm.create_model("efficientnet_b0", pretrained=True,
                              num_classes=num_classes)
    elif model_name == "ViTTiny":
        m = timm.create_model("vit_tiny_patch16_224", pretrained=True,
                              num_classes=num_classes, img_size=IMG_SIZE)
    return m.to(DEVICE)

# ── Training loop ────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE)
        labels = labels.squeeze().long().to(DEVICE)

        # FIX: Convert multi-label to multi-class if necessary
        if labels.ndim > 1:
            labels = labels.argmax(dim=1)

        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.squeeze().long().to(DEVICE)

            # FIX: Convert multi-label to multi-class if necessary
            if labels.ndim > 1:
                labels = labels.argmax(dim=1)

            out    = model(imgs)
            correct += (out.argmax(1) == labels).sum().item()
            total   += imgs.size(0)
    return correct / total

# ── Main training block ──────────────────────────────────────
MODEL_NAMES  = ["ResNet18", "EfficientNetB0", "ViTTiny"]
trained_info = {}   # {dataset_name: {model_name: {model, num_classes}}}

for ds_label, ds_flag in DATASETS.items():
    print(f"\n{'='*55}\n📂 Dataset: {ds_label}\n{'='*55}")
    info       = INFO[ds_flag]
    num_classes = len(info["label"])
    task        = info["task"]
    in_channels = info["n_channels"]

    DataClass = getattr(medmnist, info["python_class"])

    # ChestMNIST is multi-label → treat as binary per-class here
    # we keep it multi-class for simplicity (argmax prediction)
    train_ds = DataClass(split="train", transform=train_transform,
                         download=True, size=IMG_SIZE, as_rgb=True)
    val_ds   = DataClass(split="val",   transform=val_transform,
                         download=True, size=IMG_SIZE, as_rgb=True)
    test_ds  = DataClass(split="test",  transform=val_transform,
                         download=True, size=IMG_SIZE, as_rgb=True)

    # Subset for speed (keep 5k train / 1k val / 1k test)
    def subset(ds, n):
        idx = torch.randperm(len(ds))[:n].tolist()
        return Subset(ds, idx)

    train_loader = DataLoader(subset(train_ds, 5000), batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=NUM_WORKERS)
    val_loader   = DataLoader(subset(val_ds,  1000), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(subset(test_ds, 1000), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS)

    trained_info[ds_label] = {}

    for model_name in MODEL_NAMES:
        print(f"\n  🔧 Training {model_name} on {ds_label}...")
        model     = build_model(model_name, num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LR)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

        best_val_acc = 0
        ckpt_path = f"checkpoints/{ds_label}_{model_name}.pt"

        for epoch in range(1, EPOCHS + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
            val_acc          = evaluate(model, val_loader)
            scheduler.step()
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), ckpt_path)
            if epoch % 2 == 0 or epoch == EPOCHS:
                print(f"    Epoch {epoch:02d} | loss={tr_loss:.4f} "
                      f"| tr_acc={tr_acc:.3f} | val_acc={val_acc:.3f}")

        # Load best weights
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        test_acc = evaluate(model, test_loader)
        print(f"  ✅ {model_name} test acc: {test_acc:.4f}")

        trained_info[ds_label][model_name] = {
            "model":       model,
            "num_classes": num_classes,
            "test_acc":    test_acc,
            "test_loader": test_loader,
            "test_ds":     test_ds,
        }

print("\n🎉 All models trained and saved.")

In [ ]:
# ============================================================
# CELL 4 — Corruption pipeline
#           Returns a corrupted tensor given a clean tensor.
# ============================================================
import cv2

def corrupt_image(img_tensor: torch.Tensor,
                  corruption: str,
                  severity: float = 0.3) -> torch.Tensor:
    """
    img_tensor : (3, H, W)  float32 in ~[-1,1] (normalised)
    Returns    : same shape, same dtype, on CPU
    """
    if corruption == "clean":
        return img_tensor

    # Denormalize → uint8 numpy (H,W,3)
    img = img_tensor.cpu().permute(1, 2, 0).numpy()
    img = (img * 0.5 + 0.5).clip(0, 1)
    img_uint8 = (img * 255).astype(np.uint8)

    if corruption == "gaussian_noise":
        noise = np.random.normal(0, severity * 60,
                                 img_uint8.shape).astype(np.int16)
        img_uint8 = np.clip(img_uint8.astype(np.int16) + noise,
                            0, 255).astype(np.uint8)

    elif corruption == "motion_blur":
        k = max(3, int(severity * 15))
        if k % 2 == 0: k += 1
        kernel = np.zeros((k, k))
        kernel[k // 2, :] = 1.0 / k
        img_uint8 = cv2.filter2D(img_uint8, -1, kernel)

    elif corruption == "contrast_shift":
        factor = 1.0 + severity * 2.0   # up to 3x contrast
        img_f  = img_uint8.astype(np.float32)
        mean   = img_f.mean()
        img_uint8 = np.clip((img_f - mean) * factor + mean,
                            0, 255).astype(np.uint8)

    elif corruption == "brightness_shift":
        delta = int(severity * 100)
        img_uint8 = np.clip(img_uint8.astype(np.int16) + delta,
                            0, 255).astype(np.uint8)

    elif corruption == "salt_pepper":
        out  = img_uint8.copy()
        n    = int(severity * img_uint8.size * 0.1)
        coords_salt = [np.random.randint(0, i, n) for i in img_uint8.shape[:2]]
        coords_pepp = [np.random.randint(0, i, n) for i in img_uint8.shape[:2]]
        out[coords_salt[0], coords_salt[1]] = 255
        out[coords_pepp[0], coords_pepp[1]] = 0
        img_uint8 = out

    # Re-normalize → tensor
    img_back = img_uint8.astype(np.float32) / 255.0
    img_back = (img_back - 0.5) / 0.5
    return torch.tensor(img_back, dtype=torch.float32).permute(2, 0, 1)

# Quick smoke-test ─────────────────────────────────────────────
sample_path = list(trained_info.values())[0]
sample_ds   = list(list(trained_info.values())[0].values())[0]["test_ds"]
raw_img, _  = sample_ds[0]

fig, axes = plt.subplots(1, len(CORRUPTION_TYPES), figsize=(16, 3))
for ax, c in zip(axes, CORRUPTION_TYPES):
    corrupted = corrupt_image(raw_img, c)
    show = (corrupted.permute(1,2,0).numpy() * 0.5 + 0.5).clip(0,1)
    ax.imshow(show)
    ax.set_title(c.replace("_", "\n"), fontsize=9)
    ax.axis("off")
plt.suptitle("Corruption preview", fontweight="bold")
plt.tight_layout()
plt.savefig("results/corruption_preview.png", dpi=120)
plt.show()
print("✅ Corruption engine ready.")

In [ ]:
# ============================================================
# CELL 5 — Fixed: saliency map generation with diagnostics
# ============================================================

from captum.attr import LayerGradCam, IntegratedGradients
import torch.nn.functional as F
import numpy as np
import pandas as pd
import random
from tqdm import tqdm
import torch
import shap
from lime import lime_image
from collections import defaultdict

N_EXPLAIN = 50

# ── XAI failure tracker ──────────────────────────────────────
failure_log = defaultdict(int)   # {(model, xai): count}

def log_failure(model_name, xai, err):
    failure_log[(model_name, xai)] += 1

# ── GradCAM layer selector ───────────────────────────────────
def get_gradcam_layer(model, model_name):
    if model_name == "ResNet18":
        return model.layer4[-1]
    elif model_name == "EfficientNetB0":
        return model.conv_head
    elif model_name == "ViTTiny":
        # FIX: target the last transformer block directly, not norm1
        return model.blocks[-1]

# ── GradCAM ──────────────────────────────────────────────────
def explain_gradcam(model, img_tensor, target_class, layer):
    gc   = LayerGradCam(forward_func=model, layer=layer)
    attr = gc.attribute(img_tensor.unsqueeze(0).to(DEVICE),
                        target=target_class)
    attr = attr.squeeze().cpu().detach().float().numpy()
    if attr.ndim == 3:
        attr = attr.mean(0)
    # ViT produces (num_patches,) → reshape to 2D in format_map later
    return attr

# ── Integrated Gradients ─────────────────────────────────────
def explain_ig(model, img_tensor, target_class):
    ig   = IntegratedGradients(model)
    base = torch.zeros_like(img_tensor).to(DEVICE)
    attr = ig.attribute(img_tensor.unsqueeze(0).to(DEVICE),
                        baselines=base.unsqueeze(0),
                        target=target_class,
                        n_steps=50)
    attr = attr.squeeze().cpu().detach().abs().float().numpy()
    if attr.ndim == 3:
        attr = attr.mean(0)
    return attr

# ── LIME ─────────────────────────────────────────────────────
def explain_lime(model, img_tensor):
    model.eval()
    img_np = (img_tensor.permute(1,2,0).cpu().numpy() * 0.5 + 0.5).clip(0,1)

    def predict_fn(images):
        t = torch.tensor(images, dtype=torch.float32).permute(0,3,1,2)
        t = (t - 0.5) / 0.5
        with torch.no_grad():
            logits = model(t.to(DEVICE))
        return F.softmax(logits, dim=1).cpu().numpy()

    explainer = lime_image.LimeImageExplainer(verbose=False)
    exp = explainer.explain_instance(
        img_np.astype(np.double),
        predict_fn,
        top_labels=1,
        hide_color=0,
        num_samples=300,     # FIX: raised from 200 → reduces flat-mask failures
        batch_size=32,
    )
    top_label = exp.top_labels[0]
    # FIX: use positive_only=False to avoid all-zero masks when
    # no positive superpixels are found
    img_exp, mask = exp.get_image_and_mask(
        top_label,
        positive_only=False,
        num_features=10,
        hide_rest=False
    )
    attr = np.abs(mask).astype(np.float32)

    # FIX: if mask is still flat, fall back to raw segment weights
    if attr.max() - attr.min() < 1e-8:
        segments = exp.segments
        weights  = dict(exp.local_exp[top_label])
        attr     = np.vectorize(lambda s: weights.get(s, 0.0))(segments)
        attr     = np.abs(attr).astype(np.float32)

    return attr

# ── SHAP ─────────────────────────────────────────────────────
def explain_shap(model, img_tensor, background_batch):
    model.eval()
    bg  = background_batch.to(DEVICE)
    inp = img_tensor.unsqueeze(0).to(DEVICE)
    e   = shap.DeepExplainer(model, bg)
    sv  = e.shap_values(inp)

    with torch.no_grad():
        pred = model(inp).argmax(1).item()

    attr = np.abs(sv[pred] if isinstance(sv, list) else sv).squeeze()
    if attr.ndim == 3:
        attr = attr.mean(0)
    return attr.astype(np.float32)

# ── Background batch for SHAP ─────────────────────────────────
def get_background(test_ds, n=10):
    idx  = random.sample(range(len(test_ds)), n)
    imgs = [test_ds[i][0] for i in idx]
    return torch.stack(imgs)

# ── Label handler ─────────────────────────────────────────────
def parse_label(label):
    if isinstance(label, torch.Tensor):
        label = label.squeeze()
        return int(label.argmax()) if label.numel() > 1 else int(label.item())
    elif isinstance(label, np.ndarray):
        label = label.squeeze()
        return int(label.argmax()) if label.size > 1 else int(label.item())
    return int(label)

# ── Zero-map sentinel (do NOT use np.zeros — use this) ────────
def make_sentinel(shape=(IMG_SIZE, IMG_SIZE)):
    """
    Returns a tiny-noise map instead of all-zeros.
    Prevents spearmanr NaN from constant arrays while
    clearly indicating a failed explanation.
    """
    return np.random.uniform(0, 1e-6, shape).astype(np.float32)

# ── Main explanation loop ─────────────────────────────────────
explanations = {}
pred_records = []

for ds_label in DATASETS:
    explanations[ds_label] = {}
    print(f"\n{'='*55}\n📂 {ds_label}\n{'='*55}")

    for model_name, info in trained_info[ds_label].items():
        print(f"\n  🔍 {model_name}")
        model    = info["model"].eval()
        test_ds  = info["test_ds"]
        bg_batch = get_background(test_ds, n=10)
        layer    = get_gradcam_layer(model, model_name)
        indices  = random.sample(range(len(test_ds)), N_EXPLAIN)

        explanations[ds_label][model_name] = {xai: {} for xai in XAI_METHODS}

        for corruption in tqdm(CORRUPTION_TYPES,
                               desc=f"    {model_name} corruptions"):
            for xai in XAI_METHODS:
                explanations[ds_label][model_name][xai][corruption] = []

            correct = 0
            for idx in indices:
                raw_img, label = test_ds[idx]
                label   = parse_label(label)
                img_c   = corrupt_image(raw_img, corruption)

                with torch.no_grad():
                    pred_class = model(img_c.unsqueeze(0).to(DEVICE)).argmax(1).item()
                correct += int(pred_class == label)

                # GradCAM
                try:
                    attr = explain_gradcam(model, img_c, pred_class, layer)
                    explanations[ds_label][model_name]["GradCAM"][corruption].append(attr)
                except Exception as ex:
                    log_failure(model_name, "GradCAM", ex)
                    explanations[ds_label][model_name]["GradCAM"][corruption].append(
                        make_sentinel())

                # Integrated Gradients
                try:
                    attr = explain_ig(model, img_c, pred_class)
                    explanations[ds_label][model_name]["IntegratedGradients"][corruption].append(attr)
                except Exception as ex:
                    log_failure(model_name, "IntegratedGradients", ex)
                    explanations[ds_label][model_name]["IntegratedGradients"][corruption].append(
                        make_sentinel())

                # LIME
                try:
                    attr = explain_lime(model, img_c)
                    explanations[ds_label][model_name]["LIME"][corruption].append(attr)
                except Exception as ex:
                    log_failure(model_name, "LIME", ex)
                    explanations[ds_label][model_name]["LIME"][corruption].append(
                        make_sentinel())

                # SHAP
                try:
                    attr = explain_shap(model, img_c, bg_batch)
                    explanations[ds_label][model_name]["SHAP"][corruption].append(attr)
                except Exception as ex:
                    log_failure(model_name, "SHAP", ex)
                    explanations[ds_label][model_name]["SHAP"][corruption].append(
                        make_sentinel())

            acc = correct / N_EXPLAIN
            pred_records.append({
                "dataset":    ds_label,
                "model":      model_name,
                "corruption": corruption,
                "accuracy":   acc,
            })

acc_df = pd.DataFrame(pred_records)
acc_df.to_csv("results/accuracy_under_shift.csv", index=False)

# ── Failure summary ───────────────────────────────────────────
print("\n✅ All explanations generated.")
if failure_log:
    print("\n⚠️  XAI failure summary (sentinel maps used):")
    for (model_name, xai), count in sorted(failure_log.items()):
        total = len(DATASETS) * N_EXPLAIN * len(CORRUPTION_TYPES)
        print(f"   {model_name} | {xai}: {count} failures "
              f"({100*count/total:.1f}% of calls)")
else:
    print("🎉 Zero XAI failures.")

print("\n📊 Accuracy under shift:")
print(acc_df.pivot_table(index=["dataset","model"],
                          columns="corruption", values="accuracy").round(3))

In [ ]:
# ============================================================
# CELL 6 — Fixed: Stability, Sufficiency, Sensitivity, Agreement
# ============================================================

def normalize_map(m: np.ndarray) -> np.ndarray:
    mn, mx = m.min(), m.max()
    if mx - mn < 1e-8:
        return np.zeros_like(m)
    return (m - mn) / (mx - mn)

def format_map(m: np.ndarray, target_shape=(IMG_SIZE, IMG_SIZE)) -> np.ndarray:
    """Force any XAI output to 2D (H,W) matching target_shape."""
    m = np.array(m, dtype=np.float32)
    if m.ndim == 1:
        import math
        if m.shape[0] % 2 != 0:
            m = m[1:]                    # drop ViT CLS token
        side = int(math.sqrt(len(m)))
        m = m.reshape((side, side)) if side * side == len(m) else np.zeros(target_shape)
    if m.ndim == 3:
        m = m.mean(0)                    # (C,H,W) → (H,W)
    if m.shape != target_shape:
        m = np.array(Image.fromarray(m).resize(
            (target_shape[1], target_shape[0]), Image.BILINEAR))
    return m

def is_valid_map(m: np.ndarray) -> bool:
    """Returns False if map is degenerate (all zeros/constant, NaN, Inf)."""
    if m is None:
        return False
    if not np.isfinite(m).all():
        return False
    if m.max() - m.min() < 1e-8:        # constant map → spearmanr undefined
        return False
    return True

def compute_stability(map_clean, map_corrupt):
    m1 = normalize_map(map_clean.astype(np.float64))
    m2 = normalize_map(map_corrupt.astype(np.float64))
    win_size = min(7, m1.shape[0], m1.shape[1])
    if win_size % 2 == 0:
        win_size -= 1
    return float(ssim(m1, m2, data_range=1.0, win_size=max(3, win_size)))

def compute_sensitivity(map_clean, map_corrupt):
    m1 = normalize_map(map_clean)
    m2 = normalize_map(map_corrupt)
    return float(np.mean(np.abs(m1 - m2)))

def compute_sufficiency(model, img_tensor, attr_map, pred_class, top_k_ratio=0.2):
    model.eval()
    H, W = attr_map.shape
    k = max(1, int(top_k_ratio * H * W))
    flat_idx = np.argsort(attr_map.ravel())[::-1][:k]
    mask = np.zeros(H * W, dtype=np.float32)
    mask[flat_idx] = 1.0
    mask_t = torch.tensor(mask.reshape(H, W)).unsqueeze(0)

    with torch.no_grad():
        orig_conf = F.softmax(
            model(img_tensor.unsqueeze(0).to(DEVICE)), dim=1
        )[0, pred_class].item()
        img_masked = img_tensor.clone() * mask_t
        mask_conf = F.softmax(
            model(img_masked.unsqueeze(0).to(DEVICE)), dim=1
        )[0, pred_class].item()

    return orig_conf - mask_conf

def compute_agreement(maps_dict, target_shape=(IMG_SIZE, IMG_SIZE)):
    """
    FIX: Only compute spearmanr between pairs where BOTH maps are valid.
    Skips degenerate (constant/zero/NaN) maps entirely.
    Returns NaN only if fewer than 2 valid maps exist.
    """
    # Step 1 — format and validate each map
    valid = {}
    for method, m in maps_dict.items():
        m_fmt = format_map(m, target_shape)
        if is_valid_map(m_fmt):
            valid[method] = normalize_map(m_fmt).ravel()

    # Step 2 — need at least 2 valid maps to compute agreement
    if len(valid) < 2:
        return np.nan

    # Step 3 — compute pairwise spearmanr only on valid pairs
    methods = list(valid.keys())
    scores, pairs = 0.0, 0
    for i in range(len(methods)):
        for j in range(i + 1, len(methods)):
            r, _ = spearmanr(valid[methods[i]], valid[methods[j]])
            if not np.isnan(r):
                scores += r
                pairs  += 1

    return scores / pairs if pairs > 0 else np.nan

# ── Main metrics loop ─────────────────────────────────────────
records = []
nan_report = []   # track where NaNs still occur for diagnosis

for ds_label in DATASETS:
    print(f"\n📂 {ds_label}")
    for model_name, info in trained_info[ds_label].items():
        print(f"  📊 {model_name}")
        model   = info["model"].eval()
        test_ds = info["test_ds"]

        for i in tqdm(range(N_EXPLAIN), desc=f"    metrics {model_name}"):
            raw_img, label = test_ds[i]

            # Handle multi-label (ChestMNIST)
            if isinstance(label, (torch.Tensor, np.ndarray)):
                label = label.squeeze()
                label = int(label.argmax()) if label.size > 1 else int(
                    label.item() if hasattr(label, 'item') else label)
            else:
                label = int(label)

            clean_img = corrupt_image(raw_img, "clean")
            with torch.no_grad():
                pred_class = model(clean_img.unsqueeze(0).to(DEVICE)).argmax(1).item()

            clean_maps = {
                xai: format_map(
                    explanations[ds_label][model_name][xai]["clean"][i]
                ) for xai in XAI_METHODS
            }

            for corruption in CORRUPTION_TYPES:
                if corruption == "clean":
                    continue

                corrupt_img = corrupt_image(raw_img, corruption)
                with torch.no_grad():
                    pred_c = model(corrupt_img.unsqueeze(0).to(DEVICE)).argmax(1).item()

                corrupt_maps = {
                    xai: format_map(
                        explanations[ds_label][model_name][xai][corruption][i]
                    ) for xai in XAI_METHODS
                }

                # Agreement computed once per sample (same for all XAI methods)
                agreement_val = compute_agreement(corrupt_maps)

                for xai in XAI_METHODS:
                    m_clean   = clean_maps[xai]
                    m_corrupt = corrupt_maps[xai]

                    stability   = compute_stability(m_clean, m_corrupt)
                    sensitivity = compute_sensitivity(m_clean, m_corrupt)
                    sufficiency = compute_sufficiency(
                        model, corrupt_img, m_corrupt, pred_c)

                    records.append({
                        "dataset":     ds_label,
                        "model":       model_name,
                        "xai":         xai,
                        "corruption":  corruption,
                        "stability":   stability,
                        "sensitivity": sensitivity,
                        "sufficiency": sufficiency,
                        "agreement":   agreement_val,
                    })

                    if np.isnan(agreement_val):
                        nan_report.append({
                            "ds": ds_label, "model": model_name,
                            "xai": xai, "corruption": corruption, "i": i,
                            "valid_maps": [
                                k for k, m in corrupt_maps.items()
                                if is_valid_map(m)
                            ]
                        })

metrics_df = pd.DataFrame(records)
metrics_df.to_csv("results/faithfulness_metrics.csv", index=False)

# ── NaN diagnostic report ─────────────────────────────────────
print(f"\n✅ Metrics saved. Total rows: {len(metrics_df)}")
print(f"   Agreement NaN rate: "
      f"{metrics_df['agreement'].isna().mean()*100:.1f}%")
print(f"   Stability NaN rate: "
      f"{metrics_df['stability'].isna().mean()*100:.1f}%")

if nan_report:
    nan_df = pd.DataFrame(nan_report)
    print(f"\n⚠️  {len(nan_report)} NaN agreement cases. Most common valid map counts:")
    print(nan_df["valid_maps"].apply(len).value_counts())
    print("\n   Which XAI methods most often produce invalid maps:")
    from collections import Counter
    all_missing = [xai for row in nan_report
                   for xai in XAI_METHODS if xai not in row["valid_maps"]]
    print(Counter(all_missing).most_common())
else:
    print("\n🎉 Zero NaN values in agreement column.")

In [ ]:
# ============================================================
# CELL 7 — Generate all figures and summary tables
#           Saved to ./results/  (download and use in paper)
# ============================================================

metrics_df = pd.read_csv("results/faithfulness_metrics.csv")
acc_df     = pd.read_csv("results/accuracy_under_shift.csv")

PALETTE = {
    "GradCAM":            "#e74c3c",
    "IntegratedGradients":"#3498db",
    "LIME":               "#2ecc71",
    "SHAP":               "#f39c12",
}
CORRUPT_ORDER = ["gaussian_noise", "motion_blur",
                  "contrast_shift", "brightness_shift", "salt_pepper"]

# ─── FIG 1: Stability heatmap (XAI × Corruption × Dataset) ───
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, ds in zip(axes, DATASETS.keys()):
    pivot = (metrics_df[metrics_df.dataset == ds]
             .groupby(["xai", "corruption"])["stability"]
             .mean()
             .unstack("corruption")[CORRUPT_ORDER])
    sns.heatmap(pivot, ax=ax, annot=True, fmt=".2f",
                cmap="RdYlGn", vmin=0, vmax=1,
                linewidths=0.5, cbar=ax == axes[-1])
    ax.set_title(f"{ds}", fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("XAI Method")
fig.suptitle("Fig 1 — Explanation Stability (SSIM) Under Distribution Shift",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results/fig1_stability_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# ─── FIG 2: Accuracy vs Stability line plot ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, ds in zip(axes, DATASETS.keys()):
    acc_sub = acc_df[(acc_df.dataset == ds) &
                     (acc_df.corruption != "clean")]
    met_sub = (metrics_df[(metrics_df.dataset == ds)]
               .groupby(["model", "corruption"])["stability"]
               .mean().reset_index())
    merged  = pd.merge(acc_sub, met_sub, on=["model","corruption"])
    for model_name, g in merged.groupby("model"):
        ax.scatter(g["accuracy"], g["stability"],
                   label=model_name, s=80, alpha=0.8)
    ax.set_xlabel("Model Accuracy under Shift")
    ax.set_ylabel("Explanation Stability (SSIM)")
    ax.set_title(ds, fontweight="bold")
    ax.legend(fontsize=8)
    ax.axhline(0.5, color="grey", linestyle="--", alpha=0.4,
               label="Stability threshold")
axes[0].legend()
fig.suptitle("Fig 2 — Accuracy vs. Explanation Stability Under Shift\n"
             "(Key finding: high accuracy ≠ stable explanations)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("results/fig2_acc_vs_stability.png", dpi=150, bbox_inches="tight")
plt.show()

# ─── FIG 3: Agreement drop under corruption ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, ds in zip(axes, DATASETS.keys()):
    sub = (metrics_df[metrics_df.dataset == ds]
           .groupby(["model", "corruption"])["agreement"]
           .mean().reset_index())
    sub = sub[sub.corruption.isin(CORRUPT_ORDER)]
    for model_name, g in sub.groupby("model"):
        ax.plot(CORRUPT_ORDER, g.set_index("corruption").loc[CORRUPT_ORDER]["agreement"],
                marker="o", label=model_name)
    ax.set_title(ds, fontweight="bold")
    ax.set_xlabel("Corruption Type")
    ax.set_ylabel("XAI Agreement (Spearman ρ)")
    ax.set_xticklabels(CORRUPT_ORDER, rotation=20, ha="right", fontsize=8)
    ax.legend(fontsize=8)
fig.suptitle("Fig 3 — Cross-XAI Agreement Under Distribution Shift",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("results/fig3_agreement_drop.png", dpi=150, bbox_inches="tight")
plt.show()

# ─── FIG 4: Sensitivity per XAI method (bar chart) ───────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, ds in zip(axes, DATASETS.keys()):
    sub = (metrics_df[metrics_df.dataset == ds]
           .groupby("xai")["sensitivity"]
           .mean().reset_index())
    bars = ax.bar(sub["xai"], sub["sensitivity"],
                  color=[PALETTE[x] for x in sub["xai"]])
    ax.set_title(ds, fontweight="bold")
    ax.set_ylabel("Mean Sensitivity (↓ better)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
fig.suptitle("Fig 4 — Mean Saliency Sensitivity Across XAI Methods",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("results/fig4_sensitivity_bar.png", dpi=150, bbox_inches="tight")
plt.show()

# ─── TABLE 1: Full summary ────────────────────────────────────
summary = (metrics_df
           .groupby(["dataset", "model", "xai"])[
               ["stability","sensitivity","sufficiency","agreement"]]
           .mean()
           .round(3))
summary.to_csv("results/table1_full_summary.csv")
print("\n📋 TABLE 1 — Full Metric Summary")
print(summary.to_string())

# ─── TABLE 2: Per-corruption breakdown ───────────────────────
table2 = (metrics_df
          .groupby(["corruption", "xai"])[
              ["stability","sensitivity"]]
          .mean()
          .round(3))
table2.to_csv("results/table2_per_corruption.csv")
print("\n📋 TABLE 2 — Per-Corruption Breakdown")
print(table2.to_string())

print("\n\n🎉 All figures and tables saved to ./results/")
print("Files generated:")
for f in sorted(os.listdir("results")):
    print(f"  📄 results/{f}")

In [ ]:
# # ============================================================
# # CELL 5 — Generate saliency maps for each model × XAI method
# #           Results stored in `explanations` dict.
# # Runtime: ~20–40 min depending on device
# # ============================================================

# from captum.attr import LayerGradCam, IntegratedGradients
# from captum.attr import visualization as viz
# import torch.nn.functional as F
# import numpy as np
# import pandas as pd
# import random
# from tqdm import tqdm
# import torch
# import torchvision.transforms as transforms
# import shap
# from lime import lime_image

# N_EXPLAIN = 50   # number of test images to explain per combo
#                   # (50 is enough for metrics, raise to 100 if RAM allows)

# def get_gradcam_layer(model, model_name):
#     """Return the last conv layer for GradCAM."""
#     if model_name == "ResNet18":
#         return model.layer4[-1]
#     elif model_name == "EfficientNetB0":
#         return model.conv_head        # timm EfficientNet
#     elif model_name == "ViTTiny":
#         return model.blocks[-1].norm1  # last transformer block norm

# def explain_gradcam(model, img_tensor, target_class, layer):
#     """Returns (H,W) numpy attribution map."""
#     # FIX applied here: Utilizing LayerGradCam instead of generic GradCam
#     gc = LayerGradCam(forward_func=model, layer=layer)
#     attr = gc.attribute(img_tensor.unsqueeze(0).to(DEVICE),
#                         target=target_class)
#     attr = attr.squeeze().cpu().detach().numpy()
#     if attr.ndim == 3:              # (C,H,W) → (H,W)
#         attr = attr.mean(0)
#     return attr

# def explain_ig(model, img_tensor, target_class):
#     """Integrated Gradients → (H,W) numpy."""
#     ig   = IntegratedGradients(model)
#     base = torch.zeros_like(img_tensor).to(DEVICE)
#     attr = ig.attribute(img_tensor.unsqueeze(0).to(DEVICE),
#                         baselines=base.unsqueeze(0),
#                         target=target_class,
#                         n_steps=50)
#     attr = attr.squeeze().cpu().detach().abs().numpy()
#     if attr.ndim == 3:
#         attr = attr.mean(0)
#     return attr

# def explain_lime(model, img_tensor):
#     """LIME → (H,W) numpy importance map."""
#     model.eval()
#     img_np = (img_tensor.permute(1,2,0).cpu().numpy() * 0.5 + 0.5).clip(0,1)

#     def predict_fn(images):
#         # images: (N, H, W, 3) float [0,1]
#         t = torch.tensor(images, dtype=torch.float32).permute(0,3,1,2)
#         t = (t - 0.5) / 0.5
#         with torch.no_grad():
#             logits = model(t.to(DEVICE))
#         return F.softmax(logits, dim=1).cpu().numpy()

#     explainer = lime_image.LimeImageExplainer(verbose=False)
#     exp = explainer.explain_instance(
#         img_np.astype(np.double),
#         predict_fn,
#         top_labels=1,
#         hide_color=0,
#         num_samples=200,          # keep low for Colab RAM
#         batch_size=32,
#     )
#     top_label = exp.top_labels[0]
#     _, mask   = exp.get_image_and_mask(
#         top_label, positive_only=True,
#         num_features=10, hide_rest=False
#     )
#     return mask.astype(np.float32)

# def explain_shap(model, img_tensor, background_batch):
#     """KernelSHAP via DeepExplainer → (H,W) numpy."""
#     model.eval()
#     bg  = background_batch.to(DEVICE)
#     inp = img_tensor.unsqueeze(0).to(DEVICE)
#     e   = shap.DeepExplainer(model, bg)
#     sv  = e.shap_values(inp)            # list[num_classes] of (1,C,H,W)
#     # take class with highest model output
#     with torch.no_grad():
#         pred = model(inp).argmax(1).item()

#     # Handle SHAP output format depending on version
#     if isinstance(sv, list):
#         attr = np.abs(sv[pred]).squeeze()   # (C,H,W)
#     else:
#         attr = np.abs(sv).squeeze()

#     if attr.ndim == 3:
#         attr = attr.mean(0)
#     return attr

# # ── Build background batch for SHAP (10 random train images) ─
# def get_background(test_ds, n=10):
#     idx   = random.sample(range(len(test_ds)), n)
#     imgs  = []
#     for i in idx:
#         img, _ = test_ds[i]
#         imgs.append(img)
#     return torch.stack(imgs)

# # ── Main explanation loop ─────────────────────────────────────
# # explanations[ds][model][xai][corruption] = list of (H,W) arrays
# explanations  = {}
# pred_records  = []   # will store accuracy per (ds, model, corruption)

# for ds_label in DATASETS:
#     explanations[ds_label] = {}
#     print(f"\n{'='*55}\n📂 {ds_label}\n{'='*55}")

#     for model_name, info in trained_info[ds_label].items():
#         print(f"\n  🔍 {model_name}")
#         model    = info["model"].eval()
#         test_ds  = info["test_ds"]
#         bg_batch = get_background(test_ds, n=10)
#         layer    = get_gradcam_layer(model, model_name)

#         # Pick N_EXPLAIN samples
#         indices = random.sample(range(len(test_ds)), N_EXPLAIN)

#         # Clean explanations first
#         explanations[ds_label][model_name] = {xai: {} for xai in XAI_METHODS}

#         for corruption in tqdm(CORRUPTION_TYPES,
#                                desc=f"    {model_name} corruptions"):
#             for xai in XAI_METHODS:
#                 explanations[ds_label][model_name][xai][corruption] = []

#             correct = 0
#             for idx in indices:
#                 raw_img, label = test_ds[idx]

#                 # FIX: Handle multi-label shape for ChestMNIST (NumPy & Tensor friendly)
#                 if isinstance(label, torch.Tensor):
#                     label = label.squeeze()
#                     label = int(label.argmax()) if label.numel() > 1 else int(label.item())
#                 elif isinstance(label, np.ndarray):
#                     label = label.squeeze()
#                     label = int(label.argmax()) if label.size > 1 else int(label.item())
#                 else:
#                     label = int(label)

#                 img_c = corrupt_image(raw_img, corruption)

#                 # Get predicted class on corrupted image
#                 with torch.no_grad():
#                     logits = model(img_c.unsqueeze(0).to(DEVICE))
#                 pred_class = logits.argmax(1).item()
#                 correct   += int(pred_class == label)

#                 # GradCAM
#                 try:
#                     attr = explain_gradcam(model, img_c, pred_class, layer)
#                     explanations[ds_label][model_name]["GradCAM"][corruption].append(attr)
#                 except Exception as ex:
#                     explanations[ds_label][model_name]["GradCAM"][corruption].append(
#                         np.zeros((IMG_SIZE, IMG_SIZE)))

#                 # Integrated Gradients
#                 try:
#                     attr = explain_ig(model, img_c, pred_class)
#                     explanations[ds_label][model_name]["IntegratedGradients"][corruption].append(attr)
#                 except:
#                     explanations[ds_label][model_name]["IntegratedGradients"][corruption].append(
#                         np.zeros((IMG_SIZE, IMG_SIZE)))

#                 # LIME
#                 try:
#                     attr = explain_lime(model, img_c)
#                     explanations[ds_label][model_name]["LIME"][corruption].append(attr)
#                 except:
#                     explanations[ds_label][model_name]["LIME"][corruption].append(
#                         np.zeros((IMG_SIZE, IMG_SIZE)))

#                 # SHAP
#                 try:
#                     attr = explain_shap(model, img_c, bg_batch)
#                     explanations[ds_label][model_name]["SHAP"][corruption].append(attr)
#                 except:
#                     explanations[ds_label][model_name]["SHAP"][corruption].append(
#                         np.zeros((IMG_SIZE, IMG_SIZE)))

#             acc = correct / N_EXPLAIN
#             pred_records.append({
#                 "dataset":    ds_label,
#                 "model":      model_name,
#                 "corruption": corruption,
#                 "accuracy":   acc,
#             })

# acc_df = pd.DataFrame(pred_records)
# acc_df.to_csv("results/accuracy_under_shift.csv", index=False)
# print("\n✅ All explanations generated. Accuracy table saved.")
# print(acc_df.pivot_table(index=["dataset","model"],
#                           columns="corruption", values="accuracy").round(3))

In [ ]:
# # ============================================================
# # CELL 6 — Compute all 4 metrics:
# #   1. Stability  (SSIM between clean vs corrupted explanation)
# #   2. Faithfulness — Sufficiency (top-k pixel deletion)
# #   3. Agreement  (Spearman rank corr across XAI methods)
# #   4. Sensitivity (mean absolute change in saliency)
# # ============================================================

# def normalize_map(m: np.ndarray) -> np.ndarray:
#     """Min-max normalize to [0,1]."""
#     mn, mx = m.min(), m.max()
#     if mx - mn < 1e-8:
#         return np.zeros_like(m)
#     return (m - mn) / (mx - mn)

# def format_map(m: np.ndarray, target_shape=(IMG_SIZE, IMG_SIZE)) -> np.ndarray:
#     """Ensures the XAI map is strictly 2D and matches the image dimensions.
#        Crucial for handling Vision Transformer (1D patch) GradCAM outputs."""
#     if m.ndim == 1:
#         import math
#         seq_len = m.shape[0]
#         if seq_len % 2 != 0:
#             m = m[1:] # discard ViT CLS token
#         side = int(math.sqrt(len(m)))
#         if side * side == len(m):
#             m = m.reshape((side, side))
#         else:
#             m = np.zeros(target_shape)

#     if m.shape != target_shape:
#         # Resize using PIL (already imported in Cell 2)
#         m = np.array(Image.fromarray(m.astype(np.float32)).resize(
#             (target_shape[1], target_shape[0]), Image.BILINEAR))
#     return m

# def compute_stability(map_clean, map_corrupt):
#     """SSIM between normalized maps. Higher = more stable."""
#     m1 = normalize_map(map_clean.astype(np.float64))
#     m2 = normalize_map(map_corrupt.astype(np.float64))

#     # Handle window sizes for very tiny shapes to prevent SSIM crash
#     win_size = min(7, m1.shape[0], m1.shape[1])
#     if win_size % 2 == 0: win_size -= 1

#     score = ssim(m1, m2, data_range=1.0, win_size=max(3, win_size))
#     return float(score)

# def compute_sensitivity(map_clean, map_corrupt):
#     """Mean absolute pixel difference. Lower = more stable."""
#     m1 = normalize_map(map_clean)
#     m2 = normalize_map(map_corrupt)
#     return float(np.mean(np.abs(m1 - m2)))

# def compute_sufficiency(model, img_tensor, attr_map,
#                          pred_class, top_k_ratio=0.2):
#     """
#     Keep only top-k% pixels (set rest to 0) and measure
#     how much the confidence drops. Lower drop = more faithful.
#     """
#     model.eval()
#     H, W = attr_map.shape
#     k    = max(1, int(top_k_ratio * H * W))

#     flat_idx   = np.argsort(attr_map.ravel())[::-1][:k]
#     mask       = np.zeros(H * W, dtype=np.float32)
#     mask[flat_idx] = 1.0
#     mask       = mask.reshape(H, W)
#     mask_t     = torch.tensor(mask).unsqueeze(0)   # (1,H,W)

#     # Original confidence
#     with torch.no_grad():
#         orig_conf = F.softmax(
#             model(img_tensor.unsqueeze(0).to(DEVICE)), dim=1
#         )[0, pred_class].item()

#     # Masked image
#     img_masked = img_tensor.clone()
#     img_masked *= mask_t                            # zero out non-salient
#     with torch.no_grad():
#         mask_conf = F.softmax(
#             model(img_masked.unsqueeze(0).to(DEVICE)), dim=1
#         )[0, pred_class].item()

#     return orig_conf - mask_conf     # lower drop = more sufficient

# def compute_agreement(maps_dict):
#     """
#     Spearman corr between all pairs of XAI methods.
#     Returns mean pairwise correlation.
#     """
#     methods = [m for m, v in maps_dict.items() if v is not None]
#     if len(methods) < 2:
#         return np.nan
#     pairs, scores = 0, 0.0
#     for i in range(len(methods)):
#         for j in range(i+1, len(methods)):
#             f1 = normalize_map(maps_dict[methods[i]]).ravel()
#             f2 = normalize_map(maps_dict[methods[j]]).ravel()
#             r, _ = spearmanr(f1, f2)
#             # FIX: Ignore undefined correlations (NaN) triggered by all-zero arrays
#             if not np.isnan(r):
#                 scores += r
#                 pairs  += 1
#     return scores / pairs if pairs else np.nan

# # ── Main metrics computation ──────────────────────────────────
# records = []

# N_EXPLAIN_IDX = random.sample(range(
#     len(list(list(trained_info.values())[0].values())[0]["test_ds"])
# ), N_EXPLAIN)

# for ds_label in DATASETS:
#     print(f"\n📂 {ds_label}")
#     for model_name, info in trained_info[ds_label].items():
#         print(f"  📊 {model_name}")
#         model   = info["model"].eval()
#         test_ds = info["test_ds"]
#         indices = list(range(N_EXPLAIN))   # reuse same indices

#         for i, idx in enumerate(tqdm(indices, desc=f"    metrics {model_name}")):
#             raw_img, label = test_ds[idx]

#             # FIX: Handle multi-label shape for ChestMNIST
#             if isinstance(label, torch.Tensor):
#                 label = label.squeeze()
#                 label = int(label.argmax()) if label.numel() > 1 else int(label.item())
#             elif isinstance(label, np.ndarray):
#                 label = label.squeeze()
#                 label = int(label.argmax()) if label.size > 1 else int(label.item())
#             else:
#                 label = int(label)

#             clean_img = corrupt_image(raw_img, "clean")

#             with torch.no_grad():
#                 pred_class = model(
#                     clean_img.unsqueeze(0).to(DEVICE)
#                 ).argmax(1).item()

#             # Clean explanation maps
#             clean_maps = {
#                 xai: explanations[ds_label][model_name][xai]["clean"][i]
#                 for xai in XAI_METHODS
#             }

#             for corruption in CORRUPTION_TYPES:
#                 if corruption == "clean":
#                     continue
#                 corrupt_img = corrupt_image(raw_img, corruption)

#                 with torch.no_grad():
#                     pred_c = model(
#                         corrupt_img.unsqueeze(0).to(DEVICE)
#                     ).argmax(1).item()

#                 corrupt_maps = {
#                     xai: explanations[ds_label][model_name][xai][corruption][i]
#                     for xai in XAI_METHODS
#                 }

#                 for xai in XAI_METHODS:
#                     # FIX: Force format to 2D before passing to metrics
#                     m_clean   = format_map(clean_maps[xai], (IMG_SIZE, IMG_SIZE))
#                     m_corrupt = format_map(corrupt_maps[xai], (IMG_SIZE, IMG_SIZE))

#                     stability   = compute_stability(m_clean, m_corrupt)
#                     sensitivity = compute_sensitivity(m_clean, m_corrupt)
#                     sufficiency = compute_sufficiency(
#                         model, corrupt_img, m_corrupt, pred_c)
#                     agreement   = compute_agreement({
#                         k: format_map(corrupt_maps[k], (IMG_SIZE, IMG_SIZE)) for k in XAI_METHODS
#                     })

#                     records.append({
#                         "dataset":     ds_label,
#                         "model":       model_name,
#                         "xai":         xai,
#                         "corruption":  corruption,
#                         "stability":   stability,
#                         "sensitivity": sensitivity,
#                         "sufficiency": sufficiency,
#                         "agreement":   agreement,
#                     })

# metrics_df = pd.DataFrame(records)
# metrics_df.to_csv("results/faithfulness_metrics.csv", index=False)
# print("\n✅ All metrics saved to results/faithfulness_metrics.csv")
# print(metrics_df.head(10))